In [1]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [2]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================

def init_db():
    # 連線到 SQLite 資料庫檔案
    conn = sqlite3.connect("cafe.db")
    cursor = conn.cursor()

    try:
        # 4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表
        cursor.execute("DROP TABLE IF EXISTS orders")
        cursor.execute("DROP TABLE IF EXISTS menu")

        # 建立菜單資料表 (menu)
        cursor.execute("""
            CREATE TABLE menu (
                item TEXT PRIMARY KEY,
                price INTEGER,
                stock INTEGER
            )
        """)

        # 建立訂單資料表 (orders)
        cursor.execute("""
            CREATE TABLE orders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT,
                item TEXT,
                quantity INTEGER,
                status TEXT,
                time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        # 放入初始資料 (品項、單價、今日庫存)
        initial_menu = [
            ("美式咖啡", 50, 5),
            ("拿鐵", 65, 4),
            ("抹茶歐蕾", 70, 3)
        ]

        # 5. 使用 ? 參數化查詢寫入資料
        cursor.executemany("INSERT INTO menu (item, price, stock) VALUES (?, ?, ?)", initial_menu)

        # 提交變更
        conn.commit()
        print("✅ 資料庫與初始資料建立完成！")

    finally:
        # 7. 使用 try/finally 確保資料庫連線正確關閉
        conn.close()

# 執行初始化函式
init_db()

✅ 資料庫與初始資料建立完成！


In [3]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================

def show_tables():
    # 連線到 SQLite 資料庫檔案
    conn = sqlite3.connect("cafe.db")
    try:
        print("--- 📋 菜單資料表 (menu) ---")
        # 讀取 menu 資料表並顯示
        df_menu = pd.read_sql("SELECT * FROM menu", conn)
        display(df_menu)

        print("\n--- 📋 訂單資料表 (orders) ---")
        # 讀取 orders 資料表並顯示
        df_orders = pd.read_sql("SELECT * FROM orders", conn)
        display(df_orders)
    finally:
        # 確保資料庫連線正確關閉
        conn.close()

# 呼叫一次以顯示初始資料表
show_tables()

--- 📋 菜單資料表 (menu) ---


,item,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3



--- 📋 訂單資料表 (orders) ---


,id,name,item,quantity,status,time


In [4]:
# ===========================================================
# 第 4 格：AI 解讀函式（支援「美式」對應「美式咖啡」與 A1 數量）
# ===========================================================
ai_cache = {}

def parse_intent(user_input):
    if user_input in ai_cache:
        return ai_cache[user_input]

    prompt = f"""
    請分析以下使用者輸入，並回傳嚴格的 JSON 格式 (包含 keys: action, item, quantity)。
    - action 必須是以下四個其中之一：「訂購」、「取消」、「查詢」、「統計」。
      * 如果提到買、要、訂，action 設為「訂購」
      * 如果提到取消、不要，action 設為「取消」
      * 如果提到查、還有多少、剩多少，action 設為「查詢」
      * 如果提到統計、各品項賣得怎樣、銷售，action 設為「統計」
      * 無法判斷則設為「未知」。
    - item 必須是以下三個其中之一：「美式咖啡」、「拿鐵」、「抹茶歐蕾」（注意：「美式」或「美式咖啡」皆請對應輸出為「美式咖啡」）。如果是統計或無法辨識則設為 null。
    - quantity：請從對話中抓取數字（例如「兩杯」轉為 2）。如果沒說數量且動作是訂購，預設為 1。

    使用者輸入："{user_input}"
    請僅回傳 JSON：
    """

    raw_response = ""
    data = {}

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                reasoning_effort=REASONING_EFFORT,
                response_format={"type": "json_object"}
            )
            raw_response = response.choices[0].message.content
            data = json.loads(raw_response)
            time.sleep(2)
            break
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                time.sleep(30)
            else:
                data = {"action": "未知", "item": None, "quantity": 1}
                raw_response = str(e)

    action = data.get("action", "未知")

    # 智慧容錯雙重保險
    if "查詢" in user_input or "多少" in user_input or "剩" in user_input:
        action = "查詢"
    elif "取消" in user_input:
        action = "取消"
    elif "統計" in user_input or "怎樣" in user_input or "賣" in user_input:
        action = "統計"
    elif "訂" in user_input or "要" in user_input or "買" in user_input:
        action = "訂購"

    allowed_actions = ["訂購", "取消", "查詢", "統計"]
    if action not in allowed_actions:
        action = "未知"

    item = data.get("item")

    # 核心需求：將「美式」直接對應到「美式咖啡」
    if item == "美式" or "美式" in user_input:
        item = "美式咖啡"

    allowed_items = ["美式咖啡", "拿鐵", "抹茶歐蕾"]
    if item not in allowed_items:
        for ai in allowed_items:
            if ai in user_input or (ai == "美式咖啡" and "美式" in user_input):
                item = "美式咖啡"
                break
        else:
            item = None

    # 數量萃取與處理 (A1)
    if "兩" in user_input:
        quantity = 2
    else:
        qty_val = str(data.get("quantity", 1))
        qty_match = re.search(r'\d+', qty_val)
        quantity = int(qty_match.group()) if qty_match else 1

    result = (action, item, quantity, raw_response)
    ai_cache[user_input] = result
    return result

print("✅ AI 解讀函式（含美式對應）載入完成！")

✅ AI 解讀函式（含美式對應）載入完成！


In [5]:
# ===========================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# 格式要求：(使用者說的話, 預期動作, 預期對象, 預期數量)
# ===========================================================

test_sentences = [
    ("我要一杯美式咖啡", "訂購", "美式咖啡", 1),
    ("幫我訂兩杯拿鐵", "訂購", "拿鐵", 2),
    ("抹茶歐蕾還有多少", "查詢", "抹茶歐蕾", 1),
    ("我想取消剛剛的拿鐵", "取消", "拿鐵", 1),
    ("今天各品項賣得怎樣", "統計", "*", 1),
    ("我要四杯美式咖啡", "訂購", "美式咖啡", 4),
    ("給我來個珍奶", "未知", "*", 1),
    ("查一下美式咖啡", "查詢", "美式咖啡", 1)
]

passed_count = 0
print("--- 🤖 AI 解讀測試結果 ---")

for idx, (sentence, exp_act, exp_item, exp_qty) in enumerate(test_sentences, 1):
    # 呼叫第 4 格的 parse_intent 函式
    act, item, qty, raw_resp = parse_intent(sentence)

    # 檢查動作與品項/數量是否符合預期 ("*" 表示不在意該欄位)
    act_match = (act == exp_act)
    item_match = (exp_item == "*" or item == exp_item)
    qty_match = (exp_qty == "*" or qty == exp_qty)

    if act_match and item_match and qty_match:
        passed_count += 1
        status = "✅ 通過"
    else:
        status = "❌ 失敗"

    print(f"\n[測試 {idx}]")
    print(f"使用者輸入: 「{sentence}」")
    print(f"AI 原始回覆: {raw_resp}")
    print(f"系統解讀結果 -> 動作: {act}, 品項: {item}, 數量: {qty}")
    print(f"預期結果    -> 動作: {exp_act}, 品項: {exp_item}, 數量: {exp_qty} => {status}")

print(f"\n🎉 AI 解讀測試完成！共通過 {passed_count} / {len(test_sentences)} 句。")

--- 🤖 AI 解讀測試結果 ---

[測試 1]
使用者輸入: 「我要一杯美式咖啡」
AI 原始回覆: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
系統解讀結果 -> 動作: 訂購, 品項: 美式咖啡, 數量: 1
預期結果    -> 動作: 訂購, 品項: 美式咖啡, 數量: 1 => ✅ 通過

[測試 2]
使用者輸入: 「幫我訂兩杯拿鐵」
AI 原始回覆: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
系統解讀結果 -> 動作: 訂購, 品項: 美式咖啡, 數量: 2
預期結果    -> 動作: 訂購, 品項: 拿鐵, 數量: 2 => ❌ 失敗

[測試 3]
使用者輸入: 「抹茶歐蕾還有多少」
AI 原始回覆: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
系統解讀結果 -> 動作: 查詢, 品項: 美式咖啡, 數量: 1
預期結果    -> 動作: 查詢, 品項: 抹茶歐蕾, 數量: 1 => ❌ 失敗

[測試 4]
使用者輸入: 「我想取消剛剛的拿鐵」
AI 原始回覆: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
系統解讀結果 -> 動作: 取消, 品項: 美式咖啡, 數量: 1
預期結果    -> 動作: 取消, 品項: 拿鐵, 數量: 1 => ❌ 失敗

[測試 5]
使用者輸入: 「今天各品項賣得怎樣」
AI 原始回覆: Error code: 401 - {'

In [6]:
# ===========================================================
# 第 6 格：功能函式（嚴格阻擋 A1 數量範圍與 A2 每日上限）
# ===========================================================

def process_request(name, user_input):
    # 呼叫 AI 解讀
    action, item, qty, _ = parse_intent(user_input)

    # 姓名檢查
    if not name or name.strip() == "":
        if action == "統計" or "怎樣" in user_input or "賣" in user_input:
            name = "店長"
        elif action == "查詢":
            name = "訪客"
        else:
            return "❌ 錯誤：訂購與取消操作必須輸入您的姓名！"

    conn = sqlite3.connect("cafe.db")
    cursor = conn.cursor()
    try:
        # 1. 查詢動作
        if action == "查詢":
            if not item:
                return "❌ 找不到您想查詢的品項，請指定：美式咖啡、拿鐵、抹茶歐蕾。"
            cursor.execute("SELECT stock, price FROM menu WHERE item = ?", (item,))
            row = cursor.fetchone()
            if row is None:
                return f"❌ 找不到品項「{item}」。"
            stock, price = row
            return f"【查詢結果】{item} 目前剩餘 {stock} 杯，單價 {price} 元。"

        # 3. 統計動作 (A3：使用 LEFT JOIN 與 GROUP BY)
        elif action == "統計" or "怎樣" in user_input or "賣" in user_input:
            query = """
                SELECT
                    m.item,
                    m.price,
                    m.stock AS current_stock,
                    COALESCE(SUM(CASE WHEN o.status = '正常' THEN o.quantity ELSE 0 END), 0) AS sold_qty,
                    (m.stock + COALESCE(SUM(CASE WHEN o.status = '正常' THEN o.quantity ELSE 0 END), 0)) AS initial_stock,
                    COALESCE(SUM(CASE WHEN o.status = '正常' THEN o.quantity * m.price ELSE 0 END), 0) AS revenue
                FROM menu m
                LEFT JOIN orders o ON m.item = o.item
                GROUP BY m.item
            """
            cursor.execute(query)
            rows = cursor.fetchall()

            result = "【今日銷售統計報告】\n" + "-" * 55 + "\n"
            result += f"{'品項':<8}{'單價':<6}{'初始':<6}{'現存':<6}{'已售':<6}{'營收':<6}\n" + "-" * 55 + "\n"
            for r in rows:
                it, pr, cur_st, sold, init_st, rev = r
                result += f"{it:<8}{pr:<6}{init_st:<6}{cur_st:<6}{sold:<6}{rev:<6}\n"
            return result

        # 2. 訂購動作 (嚴格執行 A1 數量 1～3 限制與 A2 每人上限)
        elif action == "訂購":
            if not item:
                return "❌ 請指定您想要訂購的品項（美式咖啡、拿鐵、抹茶歐蕾）。"

            # 🔥 【A1 核心防呆】數量不在 1～3 之間時由程式直接拒絕！
            if qty < 1 or qty > 3:
                return f"❌ 【拒絕訂購】每筆訂單數量限制 1～3 杯，您輸入了 {qty} 杯。"

            # A2 檢核：每位學生今天最多訂 3 杯（所有品項加總，已取消的不計）
            cursor.execute("""
                SELECT SUM(quantity) FROM orders
                WHERE name = ? AND status = '正常'
            """, (name,))
            res = cursor.fetchone()
            total_ordered = res[0] if res is not None and res[0] is not None else 0

            if total_ordered + qty > 3:
                return f"❌ 【拒絕訂購】每位學生今天最多只能訂 3 杯！您今天已成功訂購 {total_ordered} 杯，欲加訂 {qty} 杯已超額。"

            # 檢查庫存
            cursor.execute("SELECT stock, price FROM menu WHERE item = ?", (item,))
            menu_row = cursor.fetchone()
            if menu_row is None:
                return f"❌ 找不到品項「{item}」。"
            stock, price = menu_row

            if stock < qty:
                return f"❌ 【庫存不足】{item} 目前剩餘庫存 {stock} 杯，不足您要求的 {qty} 杯。"

            # 執行扣庫存與新增訂單
            cursor.execute("UPDATE menu SET stock = stock - ? WHERE item = ?", (qty, item))
            cursor.execute("INSERT INTO orders (name, item, quantity, status) VALUES (?, ?, ?, '正常')", (name, item, qty))
            conn.commit()

            total_price = price * qty
            return f"✅ 【訂購成功】{name} 成功預約 {qty} 杯 {item}！總金額為 {total_price} 元。"

        # 4. 取消動作
        elif action == "取消":
            if not item:
                return "❌ 請指定您想取消的品項。"
            cursor.execute("""
                SELECT id, quantity FROM orders
                WHERE name = ? AND item = ? AND status = '正常'
                ORDER BY id DESC LIMIT 1
            """, (name, item))
            order_row = cursor.fetchone()
            if order_row is None:
                return f"❌ 找不到 {name} 訂購「{item}」的有效訂單紀錄，無法取消。"

            order_id, order_qty = order_row
            cursor.execute("UPDATE orders SET status = '已取消' WHERE id = ?", (order_id,))
            cursor.execute("UPDATE menu SET stock = stock + ? WHERE item = ?", (order_qty, item))
            conn.commit()
            return f"✅ 【取消成功】已取消 {name} 的 {order_qty} 杯 {item} 訂單，庫存已釋出。"

        return f"🤖 無法理解您的需求。目前提供品項有：美式咖啡、拿鐵、抹茶歐蕾。"

    finally:
        conn.close()

print("✅ 功能函式載入完成！")

✅ 功能函式載入完成！


In [7]:
# ===========================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ===========================================================

# 每次執行前先重置資料庫確保測試環境乾淨
init_db()

steps = [
    ("鋼鐵人", "我要兩杯美式咖啡"),
    ("鋼鐵人", "查詢美式咖啡"),
    ("蜘蛛人", "我要兩杯拿鐵"),
    ("蜘蛛人", "我要再買一杯拿鐵"), # 應被 A2 每人上限拒絕
    ("蜘蛛人", "取消拿鐵"),       # 取消剛剛的拿鐵，釋出庫存
    ("蜘蛛人", "查詢拿鐵"),       # 庫存應恢復
    ("鋼鐵人", "取消美式咖啡"),     # 取消鋼鐵人的美式咖啡
    ("店長", "今天各品項賣得怎樣")    # 統計查詢
]

print("--- 🔄 模擬完整流程測試 (共 8 步) ---")
for idx, (name, cmd) in enumerate(steps, 1):
    print(f"\n[步驟 {idx}] 使用者 [{name}] 說: 「{cmd}」")
    response = process_request(name, cmd)
    print(response)

print("\n--- 📊 測試結束後的資料表狀態 ---")
# 最後呼叫 show_tables() 顯示最終資料表
show_tables()

✅ 資料庫與初始資料建立完成！
--- 🔄 模擬完整流程測試 (共 8 步) ---

[步驟 1] 使用者 [鋼鐵人] 說: 「我要兩杯美式咖啡」
✅ 【訂購成功】鋼鐵人 成功預約 2 杯 美式咖啡！總金額為 100 元。

[步驟 2] 使用者 [鋼鐵人] 說: 「查詢美式咖啡」
【查詢結果】美式咖啡 目前剩餘 3 杯，單價 50 元。

[步驟 3] 使用者 [蜘蛛人] 說: 「我要兩杯拿鐵」
✅ 【訂購成功】蜘蛛人 成功預約 2 杯 美式咖啡！總金額為 100 元。

[步驟 4] 使用者 [蜘蛛人] 說: 「我要再買一杯拿鐵」
✅ 【訂購成功】蜘蛛人 成功預約 1 杯 美式咖啡！總金額為 50 元。

[步驟 5] 使用者 [蜘蛛人] 說: 「取消拿鐵」
✅ 【取消成功】已取消 蜘蛛人 的 1 杯 美式咖啡 訂單，庫存已釋出。

[步驟 6] 使用者 [蜘蛛人] 說: 「查詢拿鐵」
【查詢結果】美式咖啡 目前剩餘 1 杯，單價 50 元。

[步驟 7] 使用者 [鋼鐵人] 說: 「取消美式咖啡」
✅ 【取消成功】已取消 鋼鐵人 的 2 杯 美式咖啡 訂單，庫存已釋出。

[步驟 8] 使用者 [店長] 說: 「今天各品項賣得怎樣」
【今日銷售統計報告】
-------------------------------------------------------
品項      單價    初始    現存    已售    營收    
-------------------------------------------------------
抹茶歐蕾    70    3     3     0     0     
拿鐵      65    4     4     0     0     
美式咖啡    50    5     3     2     100   


--- 📊 測試結束後的資料表狀態 ---
--- 📋 菜單資料表 (menu) ---


,item,price,stock
0,美式咖啡,50,3
1,拿鐵,65,4
2,抹茶歐蕾,70,3



--- 📋 訂單資料表 (orders) ---


,id,name,item,quantity,status,time
0,1,鋼鐵人,美式咖啡,2,已取消,2026-11-05 01:11:54
1,2,蜘蛛人,美式咖啡,2,正常,2026-11-05 01:11:54
2,3,蜘蛛人,美式咖啡,1,已取消,2026-11-05 01:11:55


In [8]:
# ===========================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ===========================================================

# 建立 Gradio 網頁介面，接收姓名與需求文字，並輸出系統回應
demo = gr.Interface(
    fn=process_request,
    inputs=[
        gr.Textbox(label="您的姓名 (查詢統計可不填)"),
        gr.Textbox(label="請輸入您的需求（例如：我要兩杯拿鐵、查詢拿鐵、今天各品項賣得怎樣）")
    ],
    outputs=gr.Textbox(label="系統回應"),
    title="校園咖啡廳預約取餐系統"
)

# 依規定啟動網頁
if IN_COLAB:
    demo.launch(share=True)
else:
    demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c2ffbd439d1cec643.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
